# Ossia Demo Notebook

End-to-end walkthrough of the Ossia support agent **via the unified HTTP API**.
This notebook is a thin client — the actual agent runs in a server process
exposed at the base URL configured below. See `specs/SPEC.md` for the
endpoint contracts and `docs/adr/0004-unified-http-api-v1.md` for the design.

In [ ]:
import os
import httpx
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

BASE_URL = os.environ.get("OSSIA_API_URL", "http://127.0.0.1:8000")
API_KEY = os.environ.get("OSSIA_API_KEY", "")
HEADERS = {"X-API-Key": API_KEY}

r = httpx.get(f"{BASE_URL}/health", headers=HEADERS, timeout=5.0)
r.raise_for_status()
r.json()

## Inspect the loaded tools

Hits `GET /v1/tools` to show core + MCP tools with provenance.

In [ ]:
r = httpx.get(f"{BASE_URL}/v1/tools", headers=HEADERS, timeout=10.0)
r.raise_for_status()
tools = r.json()["tools"]
for t in tools[:20]:
    print(f"  [{t['source']:4s}] {t['name']}")
print(f"\n  total: {len(tools)} tools")

## Run a single chat turn

Posts to `POST /v1/chat`. Thread id is scoped server-side to the caller.

In [ ]:
r = httpx.post(
    f"{BASE_URL}/v1/chat",
    json={"message": "What are Nebius Serverless Jobs?", "thread_id": "demo-001"},
    headers=HEADERS,
    timeout=60.0,
)
r.raise_for_status()
body = r.json()
print(f"thread_id: {body['thread_id']}")
for m in body["messages"]:
    print(f"\n[{m['role']}] {m['content'][:300]}")

## Stream events for UI feedback

`POST /v1/chat/stream` returns Server-Sent Events on the v3 protocol.
Each event's `event:` field is one of `message`, `tool_call`,
`subagent`, `value`, `interrupt`, `complete`, `protocol`; the `data:`
payload is a per-kind typed object. A final `event: complete` event
is always sent; `data.interrupted=true` means the run paused on a
human-review interrupt and the client should call
`POST /v1/threads/{id}/resume` to continue.

In [ ]:
import json

with httpx.stream(
    "POST",
    f"{BASE_URL}/v1/chat/stream",
    json={"message": "How do I reset my endpoint credentials?", "thread_id": "demo-002"},
    headers=HEADERS,
    timeout=60.0,
) as r:
    r.raise_for_status()
    seen = {}
    for line in r.iter_lines():
        if not line.startswith("data:"):
            continue
        payload = json.loads(line[len("data:"):].strip())
        kind = payload.get("kind")
        seen[kind] = seen.get(kind, 0) + 1
    for kind, count in seen.items():
        print(f"  {kind:10s} x{count}")
    print(f"\n  interrupted: {seen.get('complete', 0) and 'see complete event'}")
    assert "complete" in seen, "stream must end with a kind=complete event"

## Inspect thread state after a run

`GET /v1/threads/{id}/state` returns the latest checkpointed state
(empty when no Postgres checkpointer is configured).

In [ ]:
r = httpx.get(f"{BASE_URL}/v1/threads/demo-001/state", headers=HEADERS, timeout=5.0)
r.raise_for_status()
body = r.json()
print(f"thread_id: {body['thread_id']}")
print(f"next: {body['next']}")
print(f"values keys: {sorted(body['values'].keys())}")